# 06주차 실습: Reservoir·Bloom·CMS·HLL

강의: [PDF](../pdf/week06_streaming_algorithms.pdf) · **PDF 실제 페이지 8–40** (인쇄 번호와 다를 수 있음)

## 학습 목표
정확 계산을 기준으로 확률적 자료구조의 오차와 저장 공간을 비교한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 샘플링과 Bloom false positive

In [ ]:
from labkit.sketches import reservoir, BloomFilter, CountMinSketch, HyperLogLog
from collections import Counter
sample = reservoir(iter(range(10000)), 200)
assert len(sample) == len(set(sample)) == 200
seen = list(range(1000)); absent = list(range(1000,11000))
records = []
for m in [4096,8192,16384]:
    bloom = BloomFilter(m=m, k=5)
    for x in seen: bloom.add(x)
    assert all(x in bloom for x in seen)  # false negative 없음
    records.append({'bits': m, 'payload_bytes': len(bloom.bits), 'false_positive_rate': np.mean([x in bloom for x in absent])})
display(pd.DataFrame(records))

## 빈도와 고유값 근사

In [ ]:
stream = pd.read_json(DATA/'events.jsonl', lines=True).user_id.tolist()
exact = Counter(stream)
rows = []
for width in [32,128,512]:
    cms = CountMinSketch(width=width)
    for x in stream: cms.add(x)
    errors = [cms.count(x)-v for x,v in exact.items()]
    assert min(errors) >= 0
    rows.append({'width': width, 'mean_overcount': np.mean(errors), 'payload_bytes': width*5*8})
display(pd.DataFrame(rows))
hll = HyperLogLog(p=10)
for x in stream: hll.add(x)
estimate = hll.count(); truth=len(exact)
print('HLL estimate / exact / relative error:', estimate, truth, abs(estimate-truth)/truth)
out=output_dir('week06'); pd.DataFrame(rows).to_csv(out/'cms_errors.csv',index=False)
save_json(out/'hll.json', {'estimate':estimate, 'exact':truth, 'register_bytes':len(hll.registers)})

## 확장 과제 및 제출
두 알고리즘 이상에서 파라미터·seed·입력 크기를 바꾸어 오차와 처리 시간을 비교하세요. payload bytes와 Python 객체 전체 메모리는 다릅니다. 파티션별 동일 크기 reservoir를 합치면 전체 균등 표본이 되는지 반례를 만드세요.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.